In [1]:
import pandas as pd
import numpy as np
import re
import os
from pathlib import Path

In [2]:
df = pd.read_csv("../../data/interim/02-topcv_job_filtered.csv")

In [3]:
# Lưu số lương dòng của mỗi file để so sánh
old_len = len(df)

# Thống kê số dòng của mỗi file
print(f"Số dòng của itviec_merged.csv: {old_len}")

# --- PHẦN 1: THỐNG KÊ NULL BAN ĐẦU ---
print("\n--- THỐNG KÊ NULL:  ---")
null_counts = df.isnull().sum()
null_pct = (null_counts / old_len) * 100
missing_df = pd.DataFrame({
    'Số lượng Null': null_counts,
    '% Null': null_pct.map('{:.2f}%'.format)
})
display(missing_df)

# --- PHẦN 2: XÓA DÒNG CÓ JOB_TITLE = NULL ---
df = df.dropna(subset=['job_title']).copy()

print("\n" + "-"*50)
print(f"KẾT QUẢ SAU KHI XÓA JOB_TITLE TRỐNG:")
print(f"- Merged Dataset: Đã xóa {old_len - len(df)} dòng. Còn lại {len(df)} dòng.")

Số dòng của itviec_merged.csv: 2396

--- THỐNG KÊ NULL:  ---


,Số lượng Null,% Null
url,0,0.00%
job_title,0,0.00%
company,0,0.00%
location,0,0.00%
salary,0,0.00%
experience,0,0.00%
education,0,0.00%
level,0,0.00%
employment_type,0,0.00%
industry,0,0.00%



--------------------------------------------------
KẾT QUẢ SAU KHI XÓA JOB_TITLE TRỐNG:
- Merged Dataset: Đã xóa 0 dòng. Còn lại 2396 dòng.


In [4]:
# --- THỐNG KÊ CÁC GIÁ TRỊ LƯƠNG KHÔNG CHỨA SỐ (GIÁ TRỊ NHIỄU) ---

# Hàm để lọc các giá trị không chứa chữ số
def get_non_numeric_salaries(df):
    mask_no_digits = df['salary'].str.contains(r'\d', na=True) == False
    return df[mask_no_digits]['salary']

# Thống kê cho Person 2
print(f"\n{'-'*20} CÁC LOẠI LƯƠNG 'CHỮ' {'-'*20}")
non_num = get_non_numeric_salaries(df)
salary_counts = non_num.value_counts(dropna=False)
salary_pct = (non_num.value_counts(dropna=False, normalize=True) * 100)

stats_s2 = pd.DataFrame({
    'Số lượng': salary_counts,
    'Tần suất (%)': salary_pct.map('{:.2f}%'.format)
})
display(stats_s2)


-------------------- CÁC LOẠI LƯƠNG 'CHỮ' --------------------


,Số lượng,Tần suất (%)
salary,,
Thoả thuận,1085,100.00%


In [5]:
# --- PHẦN 3: CHUẨN HÓA CỘT SALARY ---
# Biểu thức chính quy để tìm các dòng nhiễu
noise_pattern = (
    r'love it|sign in| negotiation|attractive|competitive|negotiable|'
    r'let\'s discuss|best in the market|open to negotiation|'
    r'thương lượng|thỏa thuận| thuận'
)

df['salary'] = df['salary'].fillna('Thỏa thuận')
mask = df['salary'].str.lower().str.contains(noise_pattern, na=False)
df.loc[mask, 'salary'] = 'Thỏa thuận'

print("\nĐã chuẩn hóa các dòng lương nhiễu.")

count_tt = len(df[df['salary'] == 'Thỏa thuận'])
print (f"\nSố dòng lương 'Thỏa thuận' sau khi chuẩn hóa: {count_tt} / {len(df)} dòng")
print (f"Chiếm {count_tt/len(df)*100:.2f}%" )
print(f" 5 DÒNG ĐẦU")
display(df.head())


Đã chuẩn hóa các dòng lương nhiễu.

Số dòng lương 'Thỏa thuận' sau khi chuẩn hóa: 1085 / 2396 dòng
Chiếm 45.28%
 5 DÒNG ĐẦU


,url,job_title,company,location,salary,experience,education,level,employment_type,industry,required_skills,preferred_skills,specialization,job_description,requirement,standardized_title,is_null_salary,is_thoa_thuan,is_missing_salary
0,https://www.topcv.vn/viec-lam/lap-trinh-vien-n...,Lập Trình Viên (.Net) - Không Yêu Cầu Kinh Ngh...,Công ty cổ phần công nghệ và giải pháp trực tu...,Hà Nội,8 - 20 triệu,2 năm,Đại Học trở lên,Nhân viên,Toàn thời gian,IT - Phần mềm,[],[],"['2 năm kinh nghiệm', 'Đại Học trở lên', 'Soft...",- Tham gia các dự án phần mềm của Công ty. - X...,"- Tốt nghiệp Đại học, Cao đẳng, trung tâm đào ...",Backend Developer,False,False,False
1,https://www.topcv.vn/viec-lam/mobile-app-engin...,Mobile App Engineer ( IOS / Android ) – AI Gla...,CÔNG TY TNHH VEANIX,Hà Nội,Thỏa thuận,Trên 5 năm,Đại Học trở lên,Trưởng nhóm,Toàn thời gian,IT - Phần mềm,"['Swift (ios) Hoặc Kotlin Java (android)', 'Lậ...","['Kiến Thức Về Ai Machine Learning', 'Kinh Ngh...","['Trên 5 năm kinh nghiệm', 'Đại Học trở lên', ...",Phát triển ứng dụng mobile (iOS / Android) cho...,Có kinh nghiệm phát triển mobile: iOS (Swift) ...,Mobile Developer,False,True,True
2,https://www.topcv.vn/viec-lam/backend-lead-eng...,Backend Lead Engineer,CÔNG TY TNHH LEVER SPHERE VIỆT NAM,Hà Nội,50 - 70 triệu,3 năm,Đại Học trở lên,Trưởng nhóm,Toàn thời gian,Luật,"['Thiết Kế Hệ Thống', 'System Design', 'Backen...","['Mysql/Postgresql', 'Php/Python/Java/Node.js']","['3 năm kinh nghiệm', 'Đại Học trở lên', 'Tuổi...",KEY OUTCOMES: NHỮNG KẾT QUẢ CỤ THỂ CẦN ĐẠT TRO...,NĂNG LỰC & KỸ NĂNG CẦN CÓ Expert Backend Stack...,Backend Developer,False,False,False
3,https://www.topcv.vn/viec-lam/lap-trinh-vien-n...,Lập Trình Viên Nodejs Đi Làm Ngay Tại Hà Nội,Công ty cổ phần giải pháp MKT,Hà Nội,Tới 30 triệu,1 năm,Đại Học trở lên,Nhân viên,Toàn thời gian,IT - Phần mềm,"['Git', 'SQLite', 'CI/CD', 'React', 'Node.js',...","['Native Addons (N-API)', 'Tauri', 'Publish ứn...","['1 năm kinh nghiệm', 'Đại Học trở lên', 'Tiến...",Phát triển ứng dụng desktop trên Windows sử dụ...,Tối thiểu 1 – 2 năm kinh nghiệm phát triển ứng...,Desktop Developer,False,False,False
4,https://www.topcv.vn/viec-lam/ky-su-tu-van-man...,Kỹ Sư Tư Vấn Mạng/ Presale Network Engineer,Công ty Cổ phần Viễn thông - Tin học Bưu điện ...,Hà Nội,Tới 45 triệu,3 năm,Cao Đẳng trở lên,Nhân viên,Toàn thời gian,IT - Phần mềm,[],[],"['3 năm kinh nghiệm', 'Cao Đẳng trở lên', 'Ngh...","- Làm việc với khách hàng để thu thập yêu cầu,...",- Tốt nghiệp đại học chuyên ngành Công nghệ Th...,Network Engineer,False,False,False


**Tiếp theo ta sẽ tiến hành thống nhất các nội dung ở cột salary về 1 chuẩn (đơn vị triệu đồng)**

In [6]:
def standardize_itviec_salary(salary_str):
    if pd.isna(salary_str) or salary_str == 'Thỏa thuận':
        return 0, 0, 0

    s = str(salary_str).lower().strip()
    
    # 1. Xử lý các dấu gạch ngang khác nhau (– vs -) và khoảng cách
    s = s.replace('–', '-').replace(',', '') # Xóa dấu phẩy phân cách hàng nghìn 
    
    # 2. Kiểm tra đơn vị ngoại tệ
    is_usd = any(kw in s for kw in ['usd', '$'])
    
    # 3. Bóc tách tất cả các số (bao gồm cả số nguyên và số thập phân)
    # Ví dụ: "1.5" hoặc "1200" hoặc "15.000.000"
    # Lưu ý: Vì đã xóa dấu phẩy ở bước 1, nên dấu chấm ở đây có thể là phân cách hàng nghìn (15.000.000)
    # Ta xóa dấu chấm nếu nó nằm giữa các cụm số để đưa về số nguyên thuần túy
    if '.' in s:
        # Nếu dấu chấm phân tách hàng nghìn (ví dụ 10.000.000) thì xóa
        # Nếu là số thập phân (ví dụ 1.5 triệu) thì giữ
        parts = s.split('.')
        if len(parts) > 2 or (len(parts) == 2 and len(parts[1]) > 2): 
            s = s.replace('.', '')

    numbers = re.findall(r'\d+', s)
    numbers = [float(n) for n in numbers]

    if not numbers:
        return 0, 0, 0

    min_val, max_val = 0, 0

    # 4. Xác định Logic Min/Max
    if 'up to' in s or 'upto' in s:
        min_val, max_val = 0, numbers[0]
    elif 'từ' in s or 'from' in s:
        min_val, max_val = numbers[0], 0
    elif '-' in s and len(numbers) >= 2:
        min_val, max_val = numbers[0], numbers[1]
    else:
        min_val = max_val = numbers[0]

    # 5. Chuẩn hóa đơn vị về "Triệu VNĐ"
    def to_million(val):
        if val <= 0: return 0
        # Nếu là USD
        if is_usd:
            return (val * 25000) / 1000000
        # Nếu con số khổng lồ (ví dụ 70.000.000)
        if val >= 1000000:
            return val / 1000000
        # Nếu là con số nhỏ nhưng không phải USD (ví dụ 15m, 20tr, 30) -> Nó là triệu sẵn rồi
        return val

    min_salary = to_million(min_val)
    max_salary = to_million(max_val)

    # 6. Tính Average
    if min_salary > 0 and max_salary > 0:
        avg_salary = (min_salary + max_salary) / 2
    else:
        avg_salary = max(min_salary, max_salary)

    return round(min_salary, 2), round(max_salary, 2), round(avg_salary, 2)

# --- THỰC THI TRÊN DATA ---
for df in [df]:
    results = df['salary'].apply(lambda x: standardize_itviec_salary(x))
    df[['min_salary', 'max_salary', 'avg_salary']] = pd.DataFrame(results.tolist(), index=df.index)

print(" Đã xử lý xong các cases lương đặc thù của TopCV!")
display(df[['salary', 'min_salary', 'max_salary', 'avg_salary']].head(10))

 Đã xử lý xong các cases lương đặc thù của TopCV!


,salary,min_salary,max_salary,avg_salary
0,8 - 20 triệu,8.0,20.0,14.0
1,Thỏa thuận,0.0,0.0,0.0
2,50 - 70 triệu,50.0,70.0,60.0
3,Tới 30 triệu,30.0,30.0,30.0
4,Tới 45 triệu,45.0,45.0,45.0
5,Tới 15 triệu,15.0,15.0,15.0
6,Tới 35 triệu,35.0,35.0,35.0
7,16 - 20 triệu,16.0,20.0,18.0
8,25 - 30 triệu,25.0,30.0,27.5
9,12 - 35 triệu,12.0,35.0,23.5


In [7]:
salary_col = df["salary"].astype(str).str.lower().str.strip()
df["is_null_salary"] = df["salary"].isna()
df["is_thoa_thuan"] = salary_col.str.contains("thỏa thuận",regex=True,na=False)
df["is_missing_salary"] = (
    df["is_null_salary"] |
    df["is_thoa_thuan"] |
    salary_col.isin(["", "nan", "none"])
)
print(f"Tỷ lệ dữ liệu KHÔNG có lương thực tế: {df['is_missing_salary'].mean()*100:.2f}%")
print(f"Tỷ lệ dữ liệu có lương thực tế: {(~df['is_missing_salary']).mean()*100:.2f}%")
print(f"Số dòng có lương thực tế: {(~df['is_missing_salary']).sum()} / {len(df)} dòng")
print(f"Số dòng thiếu thông tin lương: {df['is_missing_salary'].sum()} / {len(df)} dòng")
print(f"Số dòng có lương 'Thỏa thuận': {df['is_thoa_thuan'].sum()} / {len(df)} dòng")

Tỷ lệ dữ liệu KHÔNG có lương thực tế: 45.28%
Tỷ lệ dữ liệu có lương thực tế: 54.72%
Số dòng có lương thực tế: 1311 / 2396 dòng
Số dòng thiếu thông tin lương: 1085 / 2396 dòng
Số dòng có lương 'Thỏa thuận': 1085 / 2396 dòng


**Chuẩn hóa cột Location: Chỉ giữ lại tên thành phố**

Ví dụ: Ho Chi Minh

In [8]:
# Thay vì lọc và giữ phần tử ở cuối sau dấu ",", thì bên topCV là phần tử cuối ngăn cách bởi dấu "-"
def clean_location_topcv_new(loc):
    if pd.isna(loc): return "Khác"
    
    # Logic mới: Tách bằng dấu "-" và lấy phần tử cuối cùng
    # Sau đó xóa chữ "Việc làm tại" (nếu có) và xóa khoảng trắng dư thừa
    clean_loc = str(loc).split('-')[-1].replace('Việc làm tại', '').strip()
    
    # Một số dòng TopCV có thể kết thúc bằng "Hà Nội." hoặc "Hà Nội," 
    # dọn dẹp luôn các dấu câu ở cuối cho đẹp
    clean_loc = clean_loc.rstrip('.,')
    
    return clean_loc

# Áp dụng
df["location"] = df["location"].apply(clean_location_topcv_new)

# Kiểm tra kết quả
print(df["location"].value_counts())

location
Hà Nội         1643
Hồ Chí Minh     519
Đà Nẵng          48
Hải Phòng        17
Cần Thơ          15
Hưng Yên         15
Bắc Ninh         11
Đồng Nai         11
Nghệ An          11
Thanh Hóa        10
Nhật Bản          9
Phú Thọ           8
Động Hóa          6
Tây Ninh          6
Khánh Hòa         6
Vĩnh Long         6
An Giang          5
Ninh Bình         4
Cầu Giấy          4
Thủ Đức           4
Thanh Xuân        3
Thái Nguyên       3
Quảng Ninh        3
Lâm Đồng          3
Gia Lai           2
Cao Bằng          2
Bình Dương        2
Gia Lâm           2
Lào Cai           2
Hai Phong         2
Quảng Ngãi        2
Yên Phong         1
Tân Bình          1
Quận 1            1
Quận 3            1
Tương Đương       1
Động Hoá          1
Gò Vấp            1
Đo Lường          1
Cà Mau            1
Huế               1
Ho Chi Minh       1
Đức               1
Name: count, dtype: int64


In [9]:
df[df["location"].isnull()]

,url,job_title,company,location,salary,experience,education,level,employment_type,industry,...,specialization,job_description,requirement,standardized_title,is_null_salary,is_thoa_thuan,is_missing_salary,min_salary,max_salary,avg_salary


In [10]:
# Lưu file
df.to_csv("../../data/interim/03-topcv_job_filtered_cleaned.csv", index=False)